# Lecture 2: Exploratory Data Analysis & Visualization

## A hospital is about to be fined \$350,000. Is that fair?

Under the CMS Hospital Readmissions Reduction Program, hospitals with excess readmissions face penalties that can reach hundreds of thousands of dollars per year. A hospital administrator staring at a \$350,000 fine wants to know: are we actually performing worse than expected, or are we just treating sicker patients? Answering that question requires looking at the data before jumping to conclusions.

:::{.callout-important}
## Definition: Exploratory Data Analysis (EDA)

**Exploratory Data Analysis (EDA)** is the practice of examining a dataset's structure, distributions, and quirks before fitting any model. EDA is the most important step in any analysis — and the one most often skipped.
:::

Today we'll explore two real datasets: hospital readmissions and NYC Airbnb listings. Along the way, we'll find **outliers**, missing data patterns, and a lesson about adjusting for context.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['font.size'] = 12

# Load data


:::{.callout-note}
## Why "always visualize"? Anscombe's quartet

Before we dive in, here's the single best argument for plotting your data. These four datasets have *identical* means, standard deviations, correlations, and regression lines:

In [ ]:
# Anscombe's quartet: same summary stats, wildly different shapes
anscombe = sns.load_dataset('anscombe')
g = sns.lmplot(data=anscombe, x='x', y='y', col='dataset',
               col_wrap=2, ci=None, height=3,
               scatter_kws={'s': 40, 'edgecolor': 'white'})
g.set_titles('Dataset {col_name}')
g.figure.suptitle("Anscombe's Quartet: identical summary statistics, very different data", y=1.02)
plt.tight_layout()
plt.show()

Same mean of *x*. Same mean of *y*. Same correlation. Same regression line. Completely different stories. **Summary statistics can lie. Always visualize.**
:::

## Part 1: Hospital readmissions

Let's start with the CMS Hospital Readmissions Reduction Program data. Recall from Lecture 1 that this dataset tracks how often patients are readmitted within 30 days, for six medical conditions, across about 3,000 U.S. hospitals.

The EDA checklist is always the same: (1) **Shape** — how many rows and columns? (2) **Types** — what are the column types? Any surprises? (3) **Distributions** — what does each variable look like? (4) **Missing data** — how much, and why? (5) **Relationships** — how do variables relate to each other?

In [ ]:
hrrp = pd.read_csv('https://raw.githubusercontent.com/stanford-mse-125/book/main/data/hospital-readmissions/hrrp_full.csv')
print(f"Shape: {hrrp.shape}")
print(f"Columns: {list(hrrp.columns)}")

In [ ]:
# Always start here: what does the data look like?
hrrp.head()

In [ ]:
# Quick summary of numeric columns
hrrp[['Excess Readmission Ratio', 'Predicted Readmission Rate',
      'Expected Readmission Rate']].describe()

### Shape, types, and first impressions

Every EDA starts the same way: How many rows? How many columns? What types are they? Are there surprises?

In [ ]:
# Data types and non-null counts
hrrp.info()

Scan the `Dtype` column in the output. Most columns are `float64` (numbers), but `Number of Readmissions` is `object` (string). That's our first clue that something is weird. And `Number of Discharges` has fewer non-null values than other columns. Let's investigate.

### Distributions: what does "normal" look like?

The **Excess Readmission Ratio** is the key metric. It's the ratio of a hospital's predicted readmission rate to its expected rate (ERR = predicted / expected). A value of 1.0 means a hospital's readmission rate matches what's expected given its **patient mix** (the age, health conditions, and severity of the patients it treats). Above 1.0 means more readmissions than expected.

Let's see its distribution.

In [ ]:
# Overall distribution
fig, ax = plt.subplots(figsize=(8, 5))
hrrp['Excess Readmission Ratio'].dropna().hist(bins=50, ax=ax, edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('All Conditions Combined')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution by condition
fig, ax = plt.subplots(figsize=(8, 5))
for condition in hrrp['Measure Name'].unique():
    subset = hrrp[hrrp['Measure Name'] == condition]['Excess Readmission Ratio'].dropna()
    label = condition.replace('READM-30-', '').replace('-HRRP', '')
    ax.hist(subset, bins=30, alpha=0.5, label=label, edgecolor='white')
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('By Condition')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The distributions look roughly symmetric and unimodal, centered near 1.0 (though with a slight right skew — we'll revisit what "normal" really means in Lecture 8). But notice the spread differs by condition — heart failure (HF) has more variation than hip/knee replacement.

:::{.callout-tip}
## Think About It
Why might some conditions have more variability in hospital performance than others?
:::

### Missing data: is it random?

Now EDA gets interesting. How much of the data do you think is missing? 1%? 10%? 50%? Let's count.

In [ ]:
# Missing data summary
missing = hrrp.isnull().sum()
missing_pct = (missing / len(hrrp) * 100).round(1)
pd.DataFrame({'Missing': missing, '% Missing': missing_pct}).query('Missing > 0')

About 15% of `Excess Readmission Ratio` and `Predicted Readmission Rate` are missing. But is this random?

Let's check the `Number of Readmissions` column — the one stored as a string.

In [ ]:
# What non-numeric values are in Number of Readmissions?
hrrp['Number of Readmissions'].value_counts().tail(5)

In [ ]:
# How many "Too Few to Report" entries, by condition?
too_few = hrrp[hrrp['Number of Readmissions'] == 'Too Few to Report']
# :, adds commas to large numbers; :.1f rounds to 1 decimal
print(f"'Too Few to Report' entries: {len(too_few):,} ({len(too_few)/len(hrrp)*100:.1f}%)")
print()
too_few_by_condition = too_few.groupby('Measure Name').size()
total_by_condition = hrrp.groupby('Measure Name').size()
pct_missing = (too_few_by_condition / total_by_condition * 100).round(1)
pd.DataFrame({'Too Few': too_few_by_condition, 'Total': total_by_condition, '% Missing': pct_missing})

In [ ]:
# Visualize the missingness pattern
fig, ax = plt.subplots(figsize=(8, 4))
pct_missing.sort_values().plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('% Suppressed ("Too Few to Report")')
ax.set_title('Missingness varies dramatically by condition')
ax.axvline(x=pct_missing.mean(), color='black', linestyle='--',
           linewidth=1, label=f'Average: {pct_missing.mean():.0f}%')
ax.legend()
plt.tight_layout()
plt.show()

The pattern in the bar chart is a critical finding. Statisticians distinguish three patterns of missingness:

:::{.callout-important}
## Definition: Patterns of Missingness

- **Missing Completely At Random (MCAR)**: no pattern at all. A lab assistant randomly spills coffee on some data sheets — the ruined records have nothing in common with each other.
- **Missing At Random (MAR)**: missingness depends on other *observed* data, but not on the missing value itself. Older patients are less likely to complete a follow-up survey, and we have each patient's age on file — so we can account for the gap.
- **Missing Not At Random (MNAR)**: missingness depends on the *unobserved value itself*. Patients with the worst health outcomes are the ones who skip the follow-up survey, and we never observe their outcomes.
:::

Here, CMS suppresses data when counts are too small to report — the data is missing precisely *due to* the small counts. This pattern is **MNAR**, the most dangerous kind: no set of observed variables can fully correct for it.

CABG (coronary artery bypass graft surgery) has ~50% missing — fewer hospitals perform this complex procedure. The hospitals with missing data are systematically different: smaller, often rural, with fewer specialized services.

:::{.callout-warning}
## Dropping missing data can introduce bias
If you drop these rows (which an AI tool would do silently), you're analyzing only the large, urban hospitals. Your conclusions would be biased.
:::

For now, we're just diagnosing — noticing what's going on. In the next lecture (Lecture 3), we'll decide what to do about it.

:::{.callout-tip}
## Think About It
If you drop all the "Too Few to Report" rows, what kind of hospitals are left in your dataset? What conclusions might change?
:::

> **Key principle**: Before you handle missing data, you must understand *why* it's missing. The "why" determines the "how."

### Contingency tables: two categorical variables at once

We've been looking at one variable at a time. A **contingency table** (or cross-tabulation) counts how often each combination of two categorical variables occurs. It's the categorical equivalent of a scatter plot.

Let's see which conditions tend to have suppressed data. We'll create a binary column for whether data was suppressed, then cross-tabulate it with the medical condition.

In [ ]:
# Create a flag for suppressed data
hrrp['suppressed'] = (hrrp['Number of Readmissions'] == 'Too Few to Report')

# Contingency table: condition vs. suppression status
ct = pd.crosstab(hrrp['Measure Name'], hrrp['suppressed'],
                 colnames=['Suppressed'])
ct.index = ct.index.str.replace('READM-30-', '').str.replace('-HRRP', '')
ct

Raw counts are useful, but **proportions** tell a clearer story. What fraction of each condition's records are suppressed?

In [ ]:
# Row proportions: what % of each condition is suppressed?
ct_pct = pd.crosstab(hrrp['Measure Name'], hrrp['suppressed'],
                     colnames=['Suppressed'], normalize='index')
ct_pct.index = ct_pct.index.str.replace('READM-30-', '').str.replace('-HRRP', '')
ct_pct.round(3)

Now the pattern pops out: about half of CABG records are suppressed, compared to only ~8% of heart failure records. The contingency table confirms the same finding we saw in the bar chart, but makes the comparison precise and easy to read across all conditions at once.

## Part 2: Airbnb listings

We've seen how missing data can hide systematic bias in healthcare. Now let's see how outliers and skewness can mislead in a completely different domain — showing that EDA principles are universal.

We'll work with 29,000+ Airbnb listings in New York City. An Airbnb host losing bookings to cheaper listings might wonder: how should I reprice? We'll answer that question in Lecture 5, once we have regression tools. For now, we focus on understanding the data itself.

In [ ]:
# Load Airbnb data
# low_memory=False prevents mixed-type warnings for large files
airbnb = pd.read_csv('https://media.githubusercontent.com/media/stanford-mse-125/book/main/data/airbnb/listings.csv', low_memory=False)
print(f"Shape: {airbnb.shape}")

# Select key columns for exploration
cols = ['name', 'neighbourhood_group_cleansed', 'neighbourhood_cleansed',
        'room_type', 'price', 'bedrooms', 'beds',
        'number_of_reviews', 'review_scores_rating', 'accommodates']
airbnb_slim = airbnb[cols].copy()

# Price is often stored as "$1,200.00" — strip symbols and convert to float
airbnb_slim['price'] = (airbnb_slim['price']
                        .replace(r'[\$,]', '', regex=True)
                        .astype(float))

airbnb_slim.head()

### The price distribution: beware the right tail

Let's look at the distribution of nightly prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw distribution
airbnb_slim['price'].hist(bins=100, ax=axes[0], edgecolor='white')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('All prices')

# Zoomed in
airbnb_slim[airbnb_slim['price'].between(1, 500)]['price'].hist(bins=50, ax=axes[1], edgecolor='white')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Prices $1–$500 (zoomed in)')

plt.tight_layout()
plt.show()

Look at the left plot: the distribution is so skewed that you can barely see anything. A few very expensive listings (some over \$10,000/night!) dominate the scale.

Now look at the zoomed-in version on the right. Most listings are \$50–\$200/night, with a peak around \$100. Much more informative.

The distribution is clearly right-skewed.

:::{.callout-tip}
## Think About It
Will the mean be higher or lower than the median? What happens if we naively compute the mean?
:::

In [ ]:
print(f"Mean price:   ${airbnb_slim['price'].mean():.0f}/night")
print(f"Median price: ${airbnb_slim['price'].median():.0f}/night")
print(f"Listings at $0: {(airbnb_slim['price'] == 0).sum()}")
print(f"Max price: ${airbnb_slim['price'].max():,.0f}")
print(f"Listings over $1000: {(airbnb_slim['price'] > 1000).sum()}")

The mean is about \$33 higher than the median. That gap is driven entirely by the right tail — a few hundred expensive listings pull the average up. If you're an Airbnb host trying to price a typical apartment, the mean is misleading. The median is a better summary here.

:::{.callout-important}
## Definition: Robust Statistics

A statistic is **robust** if extreme values barely affect it. The **median** and **interquartile range (IQR)** are robust: moving one observation to \$100,000 hardly changes either one. The **mean** and **standard deviation** are not robust: a single extreme value can shift them dramatically. When a distribution has heavy tails or outliers, robust summaries give a more reliable picture of the typical value and spread.
:::

> **Key principle**: When a distribution has a heavy tail, the mean doesn't represent a typical value. Always look at the distribution before relying on any single number.

### Log-transforming skewed data

When data is heavily right-skewed, a **log transform** can reveal patterns that are invisible on the original scale. Taking the logarithm compresses the long right tail and stretches out the bunched-up left side.

In [ ]:
# Add log-price column (filter out $0 listings first)
airbnb_pos = airbnb_slim[airbnb_slim['price'] > 0].copy()
airbnb_pos['log_price'] = np.log10(airbnb_pos['price'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw prices — heavily skewed
airbnb_pos['price'].hist(bins=100, ax=axes[0], edgecolor='white')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Raw prices (right-skewed)')

# Log-transformed — much more symmetric
airbnb_pos['log_price'].hist(bins=50, ax=axes[1], edgecolor='white', color='seagreen')
axes[1].set_xlabel('log₁₀(Price)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-transformed prices (roughly symmetric)')

plt.tight_layout()
plt.show()

The raw histogram is nearly useless — everything is piled up on the left. After the log transform, we can see the distribution is roughly bell-shaped, centered around $10^{2.1} \approx \$125$/night.

Log transforms also make relationships easier to spot. Compare price vs. number of guests on both scales:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sample = airbnb_pos.dropna(subset=['accommodates']).sample(3000, random_state=42)

axes[0].scatter(sample['accommodates'], sample['price'], alpha=0.3, s=10)
axes[0].set_xlabel('Accommodates (guests)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Raw scale: outliers dominate')

axes[1].scatter(sample['accommodates'], sample['log_price'], alpha=0.3, s=10, color='seagreen')
axes[1].set_xlabel('Accommodates (guests)')
axes[1].set_ylabel('log₁₀(Price)')
axes[1].set_title('Log scale: trend is clearer')

plt.tight_layout()
plt.show()

On the raw scale, a handful of luxury listings ($5,000+) make it hard to see any pattern. On the log scale, a clear upward trend emerges: each additional guest corresponds to roughly a multiplicative increase in price. We'll use this idea extensively when we build regression models in Lecture 5.

### Categorical variables: room types and boroughs

Not all data is numeric. Let's look at the categorical variables. So far we've used **histograms** to see the shape of a single numeric variable. Now we need different tools:

- One categorical variable → **bar chart** (count per category)
- One numeric variable across groups → **box plot** (compare distributions)
- Two numeric variables → **scatter plot** (reveal relationships)

Choosing the right plot for the question is half the battle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Room type
room_counts = airbnb_slim['room_type'].value_counts()
room_counts.plot.bar(ax=axes[0], color=sns.color_palette()[:len(room_counts)], edgecolor='white')
axes[0].set_title('Listings by Room Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Borough
borough_counts = airbnb_slim['neighbourhood_group_cleansed'].value_counts()
borough_counts.plot.bar(ax=axes[1], color=sns.color_palette()[3:8], edgecolor='white')
axes[1].set_title('Listings by Borough')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Price by room type and borough
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Price by room type (cap at $500 for visibility)
filtered = airbnb_slim[airbnb_slim['price'].between(1, 500)]
sns.boxplot(data=filtered, x='room_type', y='price', ax=axes[0])
axes[0].set_title('Price by Room Type')
axes[0].set_ylabel('Price ($)')

# Price by borough
sns.boxplot(data=filtered, x='neighbourhood_group_cleansed', y='price', ax=axes[1])
axes[1].set_title('Price by Borough')
axes[1].set_ylabel('Price ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Two patterns jump out:

1. **Entire homes/apartments** cost roughly 2x more than private rooms, which makes intuitive sense.
2. **Manhattan** is the most expensive borough. But notice the overlap — there are cheap Manhattan listings and expensive Brooklyn listings.

:::{.callout-tip}
## Think About It
If someone told you "Manhattan Airbnbs cost more," is that the full story? What if Manhattan has more entire homes and Brooklyn has more private rooms? The comparison might be confounded by room type.
:::

## Ranking hospitals: a cautionary tale

Let's go back to the hospital data for the punchline of today's lecture.

Quick reference for the three metrics we'll use:

- **Expected Readmission Rate** = what CMS predicts *should* happen given the patient mix
- **Predicted Readmission Rate** = the hospital's actual estimated rate
- **Excess Readmission Ratio (ERR)** = Predicted / Expected (above 1.0 = worse than expected)

Suppose you want to find the hospitals with the highest readmission rates. A naive approach: just rank by `Predicted Readmission Rate`.

In [ ]:
# Focus on Heart Failure (most common condition, least missing data)
hf = hrrp[hrrp['Measure Name'] == 'READM-30-HF-HRRP'].dropna(
    subset=['Predicted Readmission Rate', 'Expected Readmission Rate', 'Excess Readmission Ratio'])

print(f"Heart failure records: {len(hf):,}")
print()
print(f"Top 10 hospitals by PREDICTED readmission rate:")
top_predicted = hf.nlargest(10, 'Predicted Readmission Rate')[
    ['Facility Name', 'State', 'Predicted Readmission Rate',
     'Expected Readmission Rate', 'Excess Readmission Ratio']]
top_predicted

Now look carefully at the `Expected Readmission Rate` column. These hospitals have *high expected rates* — meaning CMS's model predicts they *should* have high readmissions, given the patients they treat.

The `Excess Readmission Ratio` tells a different story. Let's compare the two rankings.

Here's what to look for in the next plot: the expected rate is on the x-axis, the predicted rate is on the y-axis, and color shows the excess ratio. If predicted = expected, the point falls on the dashed line. Points above the line are doing *worse* than expected; points below are doing *better*.

In [ ]:
# Scatter: Predicted vs Expected rate, colored by Excess Ratio
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(hf['Expected Readmission Rate'],
                     hf['Predicted Readmission Rate'],
                     c=hf['Excess Readmission Ratio'],
                     cmap='coolwarm', alpha=0.5, s=15, vmin=0.85, vmax=1.15)
plt.colorbar(scatter, label='Excess Readmission Ratio')
lims = [hf['Expected Readmission Rate'].min(), hf['Expected Readmission Rate'].max()]
ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='Predicted = Expected')
ax.set_xlabel('Expected Readmission Rate (%)')
ax.set_ylabel('Predicted Readmission Rate (%)')
ax.set_title('Heart Failure: Are "high-rate" hospitals actually underperforming?')
ax.legend()
plt.tight_layout()
plt.show()

The scatter plot reveals a crucial insight. Hospitals with the highest *raw* readmission rates are often treating the sickest patients. Their expected rates are high too. After adjusting for patient mix (via the Excess Readmission Ratio), many of these hospitals are performing *at or below* expectations.

Conversely, some hospitals with modest raw rates are actually doing *worse than expected* given their healthier patient mix.

In [ ]:
# Make it concrete: compare rankings
hf_ranked = hf.copy()
hf_ranked['Rank by Predicted Rate'] = hf_ranked['Predicted Readmission Rate'].rank(ascending=False).astype(int)
hf_ranked['Rank by Excess Ratio'] = hf_ranked['Excess Readmission Ratio'].rank(ascending=False).astype(int)
# Rank change shows how much the naive ranking disagrees with the adjusted ranking
hf_ranked['Rank Change'] = hf_ranked['Rank by Predicted Rate'] - hf_ranked['Rank by Excess Ratio']

# Hospitals that move the most
print("Hospitals that look MUCH WORSE in naive ranking than adjusted ranking:")
print("(Rank change > 0 means the naive ranking is harsher)")
print()
big_movers = hf_ranked.nlargest(5, 'Rank Change')[
    ['Facility Name', 'State', 'Predicted Readmission Rate',
     'Excess Readmission Ratio', 'Rank by Predicted Rate', 'Rank by Excess Ratio']]
big_movers

The naive ranking unfairly penalizes these hospitals. Their high raw rates reflect the complexity of the patients they treat — after adjustment, they're performing fine.

The ranking reversal we just witnessed is a form of **confounding**: patient severity is a lurking variable that distorts the naive comparison. A hospital can rank among the worst on raw rates yet perform at or below expectations after adjusting for its patient mix. We'll study confounding formally in Lecture 18. For now, the lesson is simpler:

:::{.callout-warning}
## Don't rank before you adjust
The "worst" hospitals might just be the ones treating the hardest cases.
:::

We'll revisit this hospital data in Lecture 18 when we ask: *is this causal?* Does the penalty actually reduce readmissions, or does it just punish hospitals that serve vulnerable populations?

## Key Takeaways

- **Always look at your data before modeling.** Use `.shape`, `.head()`, `.info()`, `.describe()` as your first moves.
- **Distributions matter.** The mean can be misleading when distributions are skewed. Always plot a histogram.
- **Missing data tells a story.** Data that's Missing Not At Random (MNAR) can bias your analysis if you just drop it. Ask *why* it's missing.
- **Outliers and confounders make naive comparisons misleading.** A few extreme values can distort means and conclusions. When you compare groups without adjusting for context (patient severity, room type, etc.), you can reach the wrong conclusion.

## Study guide

### Key ideas

- **Exploratory Data Analysis (EDA)**: Getting to know your data — shape, types, distributions, missingness, relationships — before modeling.
- **Distribution**: The pattern of values a variable takes. Described by shape (symmetric, skewed), center (mean, median), and spread.
- **Histogram**: Shows the shape of one numeric variable by binning values and counting. **Box plot**: Compares distributions across groups, showing median, quartiles, and outliers. **Scatter plot**: Reveals relationships between two numeric variables. **Bar chart**: Shows counts or proportions for categorical data.
- **Contingency table**: A table summarizing counts for two categorical variables. **Row and column proportions** show the conditional distribution of one variable given the other.
- **Outlier**: An observation far from the bulk of the data. Can be real (a penthouse) or an error.
- **Robust statistics**: The median and IQR are robust — barely affected by extreme values. The mean and standard deviation are not.
- **Patterns of missingness**: **MCAR** — no pattern at all. **MAR** — missingness depends on observed variables. **MNAR** — missingness depends on the unobserved value itself, the most dangerous pattern.
- Always visualize before summarizing. Summary statistics can hide wildly different data (Anscombe's quartet).
- When a distribution is skewed, the mean doesn't represent a typical value — the median is more informative.
- Missing data is a signal, not just a nuisance. Ask *why* it's missing before deciding how to handle it.
- Naive rankings can be unfair: the "worst" hospitals may just be treating the hardest cases.

### Computational tools

- `df.shape` — number of rows and columns
- `df.head()` — first few rows
- `df.info()` — column types and non-null counts
- `df.describe()` — summary statistics for numeric columns
- `df['col'].hist()` — histogram of one column
- `df['col'].value_counts()` — frequency table for categorical data
- `pd.crosstab()` — contingency table for two categorical variables
- `sns.boxplot()` — box plot comparing groups
- `df.isnull().sum()` — count missing values per column

### For the quiz

You should be able to: (1) describe the EDA workflow and why each step matters, (2) explain the difference between MCAR, MAR, and MNAR with an example, (3) explain why the mean is misleading for skewed data, (4) interpret a histogram, box plot, and scatter plot, and (5) explain why adjusting for context matters when comparing groups.